# POV: Dream Worlds: make the 6 AI clips for free on Google Colab

**How to use (works on a phone):**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**. Accept the warning (*Run anyway*).
3. Wait. The first run downloads the model (~5 min), then each clip takes roughly 10–20 min on the free GPU.
4. At the end a file **dream_clips.zip** downloads. Send it to Claude, who edits the final 60 s video with music.

Model: Wan 2.1 Text-to-Video 1.3B (open source, Apache-2.0). Output: 6 vertical clips, 480×832, 5 s each.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: set Runtime > Change runtime type > T4 GPU'

In [ ]:
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
import torch, os, gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()   # fits the free 15 GB T4
print('model ready')

In [ ]:
PROMPTS = [
 "First-person POV riding a mountain bike along a narrow grassy ridge on a floating island high above a sea of clouds, a giant ringed planet in the sky, waterfalls falling off the island edges, golden morning light, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, dreamy photoreal, cinematic",
 "First-person POV riding a mountain bike across pale pink sand dunes between giant glowing violet crystals, two suns low on the horizon, lavender sky with a crescent moon, rainbow light refractions, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, dreamy photoreal, cinematic",
 "First-person POV riding a mountain bike on a narrow glass bridge over a turquoise ocean that curves up into the sky like the inside of a sphere, white birds flying alongside, soft clouds, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, bright airy dreamy photoreal, cinematic",
 "First-person POV riding a mountain bike on a winding dirt path through an ancient forest of giant trees at dusk, thousands of warm floating lanterns rising between the trunks, glowing blue mushrooms, fireflies, huge pale moon through the canopy, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, magical, cinematic",
 "First-person POV riding a mountain bike along a smooth frozen river at night under huge green and violet auroras, snowy pine mountains, two moons and a blue planet on the horizon, the ice mirroring the sky, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, dreamy photoreal, cinematic",
 "First-person POV riding a mountain bike along a hilltop path through purple and white wildflowers above a sea of pink and golden clouds at sunrise, towards a giant glowing planet on the horizon, black handlebars and orange gloves at the bottom of the frame, smooth forward motion, peaceful, dreamy photoreal, cinematic"
]
NEG = 'text, watermark, logo, subtitles, people faces, deformed hands, extra fingers, blurry, low quality, jpeg artifacts, flicker, static image, still frame, cartoon, worst quality'
os.makedirs('/content/dream', exist_ok=True)
for i, p in enumerate(PROMPTS, 1):
    out = f'/content/dream/clip{i}.mp4'
    if os.path.exists(out):
        print('skip', out); continue
    print(f'--- clip {i}/6 ---')
    frames = pipe(prompt=p, negative_prompt=NEG, height=832, width=480, num_frames=81,
                  guidance_scale=5.0, num_inference_steps=30,
                  generator=torch.Generator('cpu').manual_seed(100 + i)).frames[0]
    export_to_video(frames, out, fps=16)
    del frames; gc.collect(); torch.cuda.empty_cache()
    print('saved', out)

If Colab disconnects part-way, just **Run all** again: finished clips are skipped (but `/content` is wiped if the runtime resets, so download as you go if needed).

In [ ]:
!cd /content && zip -q -r dream_clips.zip dream && ls -la dream
from google.colab import files
files.download('/content/dream_clips.zip')